# Dataset EDA and Problem Formulation

Owner: Member 2

This notebook analyses the LIAR benchmark dataset before the downstream misinformation verification pipeline. It supports the report introduction, workflow/methodology, and empirical dataset description.

## Role in the Project

The project pipeline starts with LIAR political claims, then uses claim extraction, entity linking, knowledge graph reasoning, Bayesian inference, and responsible AI reflection. This notebook documents the input data quality and distribution so later results can be interpreted responsibly.

Main outputs from this notebook are saved to `02_Problem_Dataset_EDA/` and `report_assets/`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'LIAR_dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODULE_DIR = PROJECT_ROOT / '02_Problem_Dataset_EDA'
sys.path.insert(0, str(MODULE_DIR))

from run_eda import run_eda

results = run_eda(PROJECT_ROOT)
df = results['data']
summary = results['summary']

summary

{'dataset': 'LIAR',
 'source_files': ['train.tsv', 'valid.tsv', 'test.tsv'],
 'total_rows': 12791,
 'rows_by_split': {'train': 10240, 'valid': 1284, 'test': 1267},
 'label_order': ['pants-fire',
  'false',
  'barely-true',
  'half-true',
  'mostly-true',
  'true'],
 'overall_label_distribution': {'pants-fire': 1047,
  'false': 2507,
  'barely-true': 2103,
  'half-true': 2627,
  'mostly-true': 2454,
  'true': 2053},
 'duplicate_statement_count': 26,
 'duplicate_statement_percent': 0.2,
 'duplicate_id_count': 0,
 'missing_values': {'statement_id': 0,
  'label': 0,
  'statement': 0,
  'subjects': 2,
  'speaker': 2,
  'speaker_job': 3566,
  'state': 2748,
  'party': 2,
  'barely_true_count': 0,
  'false_count': 0,
  'half_true_count': 0,
  'mostly_true_count': 0,
  'pants_fire_count': 0,
  'context': 131},
 'mean_statement_words': 18.04,
 'median_statement_words': 17.0,
 'top_speakers': [{'speaker': 'barack-obama', 'count': 611},
  {'speaker': 'donald-trump', 'count': 343},
  {'speaker': '

## Dataset Schema

The LIAR TSV files contain statement ID, six-class truth label, statement text, topic/subjects, speaker metadata, party/state fields, speaker credit-history counts, and context. Blank strings are treated as missing values during EDA.

In [2]:
df.head()

,split,statement_id,label,statement,subjects,speaker,speaker_job,state,party,barely_true_count,false_count,half_true_count,mostly_true_count,pants_fire_count,context,statement_word_count,subject_count
0,train,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0,1,0,0,0,a mailer,11,1
1,train,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0,0,1,1,0,a floor speech.,24,3
2,train,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70,71,160,163,9,Denver,19,1
3,train,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,,,none,7,19,3,5,44,a news release,12,1
4,train,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,,Florida,democrat,15,9,20,19,2,an interview on CNN,10,2


## Split Sizes

This confirms the number of rows in train, validation, and test splits and checks whether the benchmark split is approximately balanced.

In [3]:
results['split_summary']

,split,row_count,unique_labels,unique_speakers,duplicate_statements,mean_statement_words,median_statement_words
0,train,10240,6,2911,17,18.01,17.0
1,valid,1284,6,662,0,17.93,17.0
2,test,1267,6,636,0,18.40,16.0


## Truth Label Distribution

The project keeps the original six LIAR labels instead of forcing them into binary true/false labels. This preserves uncertainty and nuance for downstream analysis.

In [4]:
results['label_distribution']

,split,label,count,percent
0,train,pants-fire,839,8.19
1,train,false,1995,19.48
2,train,barely-true,1654,16.15
3,train,half-true,2114,20.64
4,train,mostly-true,1962,19.16
5,train,true,1676,16.37
6,valid,pants-fire,116,9.03
7,valid,false,263,20.48
8,valid,barely-true,237,18.46
9,valid,half-true,248,19.31


## Data Quality Checks

Missing values and duplicate statements are reported rather than automatically removed. This keeps the original benchmark intact and makes preprocessing decisions transparent.

In [5]:
results['missing_values']

,column,missing_count,missing_percent
0,statement_id,0,0.00
1,label,0,0.00
2,statement,0,0.00
3,subjects,2,0.02
4,speaker,2,0.02
5,speaker_job,3566,27.88
6,state,2748,21.48
7,party,2,0.02
8,barely_true_count,0,0.00
9,false_count,0,0.00


In [6]:
df[df.duplicated(subset=['statement'], keep=False)].sort_values('statement').head(10)

,split,statement_id,label,statement,subjects,speaker,speaker_job,state,party,barely_true_count,false_count,half_true_count,mostly_true_count,pants_fire_count,context,statement_word_count,subject_count
2586,train,12055.json,mostly-true,Americans havent had a raise in 15 years.,income,hillary-clinton,Presidential candidate,New York,democrat,40,29,69,76,7,a Hillary for America ad,8,1
10716,valid,11898.json,mostly-true,Americans havent had a raise in 15 years.,"economy,jobs",hillary-clinton,Presidential candidate,New York,democrat,40,29,69,76,7,a Democratic presidential debate in Milwaukee,8,2
1582,train,4668.json,half-true,During Sherrod Browns past decade as a D.C. po...,"economy,job-accomplishments,jobs",josh-mandel,Ohio treasurer,Ohio,republican,4,5,4,5,6,a news release,36,3
4839,train,4669.json,half-true,During Sherrod Browns past decade as a D.C. po...,"economy,job-accomplishments,jobs",josh-mandel,Ohio treasurer,Ohio,republican,4,5,4,5,6,a news release,36,3
1526,train,7763.json,half-true,"Four balanced budgets in a row, with no new ta...","education,state-budget,state-finances",chris-christie,Governor of New Jersey,New Jersey,republican,10,17,27,19,8,a TV ad,38,3
2846,train,7756.json,mostly-true,"Four balanced budgets in a row, with no new ta...","job-accomplishments,jobs,state-budget,state-fi...",chris-christie,Governor of New Jersey,New Jersey,republican,10,17,27,19,8,a television ad,38,5
3339,train,11798.json,false,Martin Luther King Jr. was a Republican.,"civil-rights,history",philip-van-cleave,president of the Virginia Citizens Defense League,Virginia,organization,0,1,0,0,0,a speech.,7,2
10678,valid,5212.json,false,Martin Luther King Jr. was a Republican.,"bipartisanship,history",charlotte-bergmann,Candidate,Tennessee,republican,0,2,0,0,1,a campaign billboard along I-240.,7,2
848,train,828.json,false,"Obama says Iran is a 'tiny' country, 'doesn't ...",foreign-policy,john-mccain,U.S. senator,Arizona,republican,31,39,31,37,8,a TV ad airing in Florida,12,1
1846,train,662.json,false,"Obama says Iran is a 'tiny' country, 'doesn't ...",foreign-policy,john-mccain,U.S. senator,Arizona,republican,31,39,31,37,8,a TV ad,12,1


## Speaker, Party, Topic, and Context Analysis

These fields are useful for understanding dataset coverage and bias risk. They should not be treated as direct evidence that a claim is true or false because they may encode historical or political bias.

In [7]:
results['top_speakers']

,speaker,count
0,barack-obama,611
1,donald-trump,343
2,hillary-clinton,297
3,mitt-romney,212
4,john-mccain,189
5,scott-walker,183
6,chain-email,178
7,rick-perry,173
8,marco-rubio,153
9,rick-scott,150


In [8]:
results['top_parties']

,party,count
0,republican,5665
1,democrat,4137
2,none,2181
3,organization,264
4,independent,180
5,newsmaker,64
6,libertarian,51
7,journalist,49
8,activist,45
9,columnist,44


In [9]:
results['top_subjects']

,subject,count
0,economy,1432
1,health-care,1426
2,taxes,1218
3,federal-budget,937
4,education,926
5,jobs,899
6,state-budget,879
7,candidates-biography,805
8,elections,757
9,immigration,642


## Link to Claim Extraction

The first pipeline notebook has already produced `01_Claim_Extraction/extracted_triples_filtered.json`. This EDA notebook does not replace that step; it provides dataset context for the report and checks that the downstream handoff exists.

In [10]:
summary['claim_extraction_handoff']

{'available': True,
 'record_count': 500,
 'columns': ['raw_claim',
  'speaker',
  'label',
  'subject',
  'subject_type',
  'relation',
  'object',
  'object_type',
  'date',
  'confidence',
  'entities'],
 'label_distribution': {'half-true': 114,
  'mostly-true': 97,
  'false': 90,
  'barely-true': 81,
  'true': 79,
  'pants-fire': 39},
 'top_relations': {'say': 99,
  'be': 72,
  'have': 26,
  'support': 8,
  'vote': 8,
  'take': 5,
  'cut': 5,
  'go': 5,
  'on': 4,
  'have be': 4},
 'subject_type_distribution': {'UNKNOWN': 211,
  'PER': 162,
  'LOC': 53,
  'MISC': 39,
  'ORG': 35},
 'mean_extraction_confidence': 0.7735}

## Generated Report Assets

The reusable EDA runner writes CSV tables, PNG figures, `dataset_summary.json`, and `preprocessing_notes.md`. These files can be referenced directly in the final report.

In [11]:
summary['generated_tables'], summary['generated_figures']

(['report_assets/tables/dataset_split_summary.csv',
  'report_assets/tables/label_distribution.csv',
  'report_assets/tables/missing_values.csv',
  'report_assets/tables/top_speakers.csv',
  'report_assets/tables/top_parties.csv',
  'report_assets/tables/top_states.csv',
  'report_assets/tables/top_contexts.csv',
  'report_assets/tables/top_subjects.csv'],
 ['report_assets/figures/liar_split_sizes.png',
  'report_assets/figures/liar_label_distribution.png',
  'report_assets/figures/liar_party_distribution.png'])